In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import openpyxl

## Task 1: Data Profiling and Missing Values (~25 minutes)

### 1.1 — Profile the raw data

Compute a comprehensive profile of the dataset:

- Number of rows, columns, and memory usage
- Missing value counts and percentages for each column
- Number of unique values per column
- Basic statistics for numeric columns (min, max, mean, median, std)




In [2]:
df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")
print(f"\nMemory usage:\n{df.memory_usage(deep=True)}")
print(f"\nMissing percentage:\n{(df.isnull().sum()/len(df))*100}")
print(f"\nUnique values per column:\n{df.nunique()}")
print(f"\nStatistics for numeric columns:\n{df.describe()}")
print(f"\nMedian values:\n{df.median(numeric_only=True)}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

### 1.2 — Classify the missing data types

The two columns with significant missing values are `CustomerID` and `Description`.

For each:
- Determine whether the missingness is MNAR, MAR, or MCAR. Justify your answer with evidence (e.g., compare rows with and without missing values — do they differ systematically in other columns?).
- Decide on a strategy: deletion, imputation, or indicator column. Justify your choice.

Guiding questions:
- Are transactions with missing `CustomerID` different from those with a `CustomerID`? Check the distribution of `Country`, `Quantity`, and `UnitPrice` for both groups.
- Do transactions with missing `Description` have valid `StockCode` values? Could you recover descriptions from other rows with the same `StockCode`?


In [3]:
missing_cid=df[df["CustomerID"].isnull()]
not_missing_cid=df[df["CustomerID"].notnull()]

In [4]:
print(f"missing customer ids:")
print(missing_cid["Country"].value_counts(normalize=True))
print(f"\n not missing customer ids:") 
print(not_missing_cid["Country"].value_counts(normalize=True))

missing customer ids:
Country
United Kingdom    0.989044
EIRE              0.005264
Hong Kong         0.002132
Unspecified       0.001495
Switzerland       0.000925
France            0.000489
Israel            0.000348
Portugal          0.000289
Bahrain           0.000015
Name: proportion, dtype: float64

 not missing customer ids:
Country
United Kingdom          0.889509
Germany                 0.023339
France                  0.020871
EIRE                    0.018398
Spain                   0.006226
Netherlands             0.005828
Belgium                 0.005086
Switzerland             0.004614
Portugal                0.003638
Australia               0.003095
Norway                  0.002669
Italy                   0.001974
Channel Islands         0.001863
Finland                 0.001708
Cyprus                  0.001529
Sweden                  0.001136
Austria                 0.000986
Denmark                 0.000956
Japan                   0.000880
Poland                  0.00083

In [5]:
print(missing_cid[["Quantity","UnitPrice"]].describe())

            Quantity      UnitPrice
count  135080.000000  135080.000000
mean        1.995573       8.076577
std        66.696153     151.900816
min     -9600.000000  -11062.060000
25%         1.000000       1.630000
50%         1.000000       3.290000
75%         3.000000       5.450000
max      5568.000000   17836.460000


In [6]:
print(not_missing_cid[["Quantity","UnitPrice"]].describe())

            Quantity      UnitPrice
count  406829.000000  406829.000000
mean       12.061303       3.460471
std       248.693370      69.315162
min    -80995.000000       0.000000
25%         2.000000       1.250000
50%         5.000000       1.950000
75%        12.000000       3.750000
max     80995.000000   38970.000000


To investigate the missingness of CustomerID, the distributions of Country, Quantity, and UnitPrice were compared between transactions with and without CustomerID. Transactions with missing CustomerID are heavily concentrated in the United Kingdom (≈98.9%), while transactions with a valid CustomerID come from a wider range of countries. In addition, transactions without CustomerID have a much lower average quantity (≈2 vs ≈12) but a higher average unit price (≈8.07 vs ≈3.46). These systematic differences indicate that the missingness is related to other observed variables in the dataset rather than occurring completely at random. Therefore, the missingness of CustomerID is best classified as Missing At Random (MAR).Since CustomerID mainly identifies the customer rather than describing the transaction itself, a reasonable strategy is to remove rows with missing CustomerID or create an indicator variable to mark unknown customers.

In [7]:
missing_desc = df[df["Description"].isnull()]
missing_desc["StockCode"].isnull().sum()

np.int64(0)

In [8]:
desc_per_code = df.groupby("StockCode")["Description"].nunique()
desc_per_code.value_counts()

Description
1    3308
2     517
0     112
3      98
4      26
5       5
6       2
8       1
7       1
Name: count, dtype: int64

In [9]:
codes_with_missing_desc = missing_desc["StockCode"].unique()

valid_desc_rows = df[
    (df["StockCode"].isin(codes_with_missing_desc)) &
    (df["Description"].notnull())
]
print("\nStockCodes with missing Description that have valid descriptions elsewhere:")
print(valid_desc_rows["StockCode"].nunique())


StockCodes with missing Description that have valid descriptions elsewhere:
848


Most rows with missing Description still contain a valid StockCode. When grouping by StockCode, we see that most product codes correspond to only one unique description, and the same StockCodes often appear in other rows where the description is present. This indicates that the missing descriptions are related to another observed variable (StockCode), so the missingness is MAR (Missing At Random).

The best strategy is imputation, where missing descriptions are filled using the description associated with the same StockCode in other rows.

### 1.3 — Handle the missing values

Apply your chosen strategies. Document what you did and why. After handling missing values, print the remaining missing value counts to confirm.


In [10]:
df_=df.copy()

In [11]:
df_ = df_.dropna(subset=["CustomerID"])

In [12]:
desc_map=df_.dropna(subset=["Description"]).drop_duplicates(subset=["StockCode"])
desc_map=desc_map[["Description","StockCode"]]
df_=df_.merge(desc_map,on="StockCode",how="left",suffixes=("","_filled"))
df_["Description"]=df_["Description"].fillna(df_["Description_filled"])
df_=df_.drop(columns=["Description_filled"])
df_

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
406824,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
406825,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
406826,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
406827,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [13]:
df_.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [14]:
df_.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,406829.000000,406829,406829.000000,406829.000000
mean,12.061303,2011-07-10 16:30:57.879207424,3.460471,15287.690570
min,-80995.000000,2010-12-01 08:26:00,0.000000,12346.000000
25%,2.000000,2011-04-06 15:02:00,1.250000,13953.000000
50%,5.000000,2011-07-31 11:48:00,1.950000,15152.000000
75%,12.000000,2011-10-20 13:06:00,3.750000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,248.693370,NaN,69.315162,1713.600303


In [15]:
df_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406829 entries, 0 to 406828
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  object        
 1   StockCode    406829 non-null  object        
 2   Description  406829 non-null  object        
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[ns]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      406829 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 24.8+ MB


For CustomerID, the missingness appeared to be related to transaction characteristics (such as Country, Quantity, and UnitPrice), suggesting a MAR pattern. These rows likely correspond to anonymous or unregistered customers. Since CustomerID is not necessary for analyzing product-level transactions, rows with missing CustomerID were removed from the dataset.

For Description, most rows with missing values still had valid StockCode entries, and the same StockCodes appeared elsewhere in the dataset with valid descriptions. This indicates that the missing descriptions can be recovered from other rows with the same StockCode. Therefore, imputation was applied by filling missing descriptions using the description associated with the same StockCode.

## Task 2: Cleaning Invalid and Anomalous Records

### 2.1 — Identify cancellations

Invoices starting with "C" are cancellations. These have negative quantities and represent returns, not purchases.

- Count how many cancellation transactions exist.
- Decide whether to keep, remove, or flag them. Think about the downstream task: if you later want to predict customer churn or build a recommender, how do cancellations affect the analysis?




In [16]:
cancellations=df[df["InvoiceNo"].str.startswith("C",na=False)]
len(cancellations)

9288

In [17]:
print("Number of cancellation invoices:", cancellations["InvoiceNo"].nunique())

Number of cancellation invoices: 3836


In [18]:
df["is_cancellation"] = df["InvoiceNo"].str.startswith("C", na=False)

Invoices that start with "C" represent product returns or cancellations. These transactions usually contain negative quantities because they reverse previous purchases.

Instead of removing them, a new column (is_cancellation) was created to flag these transactions. Keeping cancellations can be useful for downstream tasks. For example, frequent returns may indicate customer dissatisfaction and could be relevant for customer churn prediction. In recommendation systems, returned products may also provide signals about products a customer did not like.

### 2.2 — Handle invalid quantities and prices

Investigate records with:
- `Quantity <= 0` (that are not cancellations)
- `UnitPrice <= 0`
- Extreme outliers in `Quantity` or `UnitPrice`

For each category of invalid record:
- Count how many exist
- Decide what to do (remove, cap, flag) and justify


In [19]:
invalid_quantity=df[(df["Quantity"]<=0)&(~df["InvoiceNo"].str.startswith("C",na=False))]
len(invalid_quantity)

1336

In [20]:
df_=df[~((df["Quantity"]<=0)&(~df["InvoiceNo"].str.startswith("C",na=False)))]

In [21]:
invalid_price=df[df["UnitPrice"]<=0]
len(invalid_price)

2517

In [22]:
df_ = df_[df_["UnitPrice"] > 0].copy()

In [23]:
Q1=df["Quantity"].quantile(0.75)
Q3=df["Quantity"].quantile(0.25)
IQR=Q3-Q1
lower=Q1-1.5*IQR
higher=Q3+1.5*IQR
outliers_quantity=df[(df["Quantity"]<lower)|(df["Quantity"]>higher)]
len(outliers_quantity)

541909

In [24]:
Q1 = df["UnitPrice"].quantile(0.25)
Q3 = df["UnitPrice"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers_price = df[(df["UnitPrice"] < lower) | (df["UnitPrice"] > upper)]
len(outliers_price)


39627

In [25]:
df["Quantity"] = df["Quantity"].clip(lower, upper)

Records with Quantity ≤ 0 that are not cancellations were identified as invalid transactions, since negative quantities should only appear in cancellation invoices. These rows were removed from the dataset.

Rows with UnitPrice ≤ 0 were also considered invalid because prices should not be zero or negative for standard transactions. These records were removed.

For extreme outliers in Quantity and UnitPrice, the IQR method was used to detect unusually large values. Instead of removing them completely, the values were capped to reduce their influence while preserving potentially valid bulk purchases or high-priced items.

### 2.3 — Clean and validate

Apply your cleaning rules. After cleaning, verify:
- No remaining negative quantities (unless you kept cancellations with a flag)
- No zero or negative prices
- Print the shape before and after cleaning


In [26]:
print("Shape before cleaning:", df.shape)
print("Shape after cleaning:", df_.shape)

Shape before cleaning: (541909, 9)
Shape after cleaning: (539392, 9)


In [27]:
print("Remaining negative quantities (non-cancellation):")
print(df_[(df_["Quantity"] <= 0) & (~df_["InvoiceNo"].str.startswith("C", na=False))].shape[0])

Remaining negative quantities (non-cancellation):
0


In [28]:
print("Remaining zero or negative prices:")
print((df_["UnitPrice"] <= 0).sum())

Remaining zero or negative prices:
0


The cleaning steps were applied to a copy of the dataset (df_) in order to preserve the original data. During cleaning, invalid records with non-positive quantities (excluding cancellations) and non-positive unit prices were removed.

To verify the results, the dataset shape was compared before and after cleaning. Additional checks confirmed that no non-cancellation transactions remain with negative quantities and that no records contain zero or negative unit prices.

## Task 3: Categorical Data Challenges

### 3.1 — Analyze the Country column

- How many unique countries are there?
- What percentage of transactions come from the top 5 countries?
- How many countries have fewer than 50 transactions?

Create a cleaned version of the `Country` column that groups rare countries (fewer than 50 transactions) into an "Other" category. Compare the number of categories before and after.




In [29]:
df["Country"].nunique()

38

In [30]:
countries=df["Country"].value_counts()
top5=df["Country"].value_counts().head(5).sum()
countries_sum=countries.sum()
perc=(top5/countries_sum)*100
print(round(perc,2))

96.74


In [31]:
rare_countries=countries[countries<50]
len(rare_countries)

6

In [32]:
df_["cleaned_country"] = df_["Country"].apply(
    lambda x: "Other" if countries[x] < 50 else x
)

In [33]:
print("Categories before:", df_["Country"].nunique())
print("Categories after:", df_["cleaned_country"].nunique())

Categories before: 38
Categories after: 33


### 3.2 — Analyze the StockCode column

`StockCode` is a high-cardinality categorical feature.

- How many unique stock codes exist?
- Are there non-product codes (e.g., postage, discounts, manual adjustments)? Identify them by looking at codes that are not purely numeric or have unusual patterns.
- Decide how to handle non-product codes for a product-level analysis.


In [34]:
df["StockCode"].nunique()

4070

In [35]:
non_numeric = df[~df["StockCode"].astype(str).str.isnumeric()]
non_numeric["StockCode"].unique()

array(['85123A', '84406B', '84029G', ..., '85179a', '90214U', '47591b'],
      shape=(1124,), dtype=object)

In [36]:
non_numeric["StockCode"].value_counts()

StockCode
85123A    2313
85099B    2159
POST      1256
85099C     960
82494L     943
          ... 
84845D       1
82545A       1
84251F       1
35603B       1
47591b       1
Name: count, Length: 1124, dtype: int64

In [37]:
df_products = df[df["StockCode"].astype(str).str.isnumeric()]

### 3.3 — Engineer a feature from Description

The `Description` column contains free text. Create a simple feature from it — for example, the word count of the description, or a flag for descriptions containing certain keywords (e.g., "SET", "PACK", "VINTAGE"). Show that your engineered feature has a meaningful relationship with `Quantity` or `UnitPrice`.


In [38]:
df["desc_word_count"] = df["Description"].astype(str).str.split().str.len()

In [39]:
df.groupby("desc_word_count")["Quantity"].mean()

desc_word_count
1    1.085018
2    3.978892
3    4.418266
4    4.312778
5    4.252328
6    4.222597
7    4.069733
8    4.079850
Name: Quantity, dtype: float64

In [40]:
df.groupby("desc_word_count")["UnitPrice"].mean()

desc_word_count
1    67.694982
2    43.260260
3     2.889057
4     3.426120
5     3.244060
6     3.017217
7     3.576106
8     1.691410
Name: UnitPrice, dtype: float64

A simple feature was engineered from the Description column by calculating the number of words in each description.
This feature captures the level of detail in product descriptions. Grouping by this feature shows differences in the average Quantity and UnitPrice, suggesting that description length may relate to how products are sold or priced.

## Task 4: Class Imbalance — Predicting High-Value Customers

### 4.1 — Engineer a binary target

Create a customer-level dataset by aggregating transactions per `CustomerID`. Compute:
- Total revenue (`Quantity * UnitPrice`)
- Number of orders (unique `InvoiceNo` values)
- Number of distinct products purchased
- Date of first and last purchase

Define a binary target: a customer is **high-value** if their total revenue is in the top 10%. Label these as `1` and the rest as `0`.

```python
# Hint: compute total revenue per customer, then use a percentile threshold
customer_revenue = df.groupby("CustomerID").apply(
    lambda x: (x["Quantity"] * x["UnitPrice"]).sum()
).reset_index(name="total_revenue")
```

In [44]:
df_["Revenue"]=df_["Quantity"]*df_["UnitPrice"]
customer = df_.groupby("CustomerID").agg(
    total_revenue=("Revenue","sum"),
    num_orders=("InvoiceNo","nunique"),
    num_products=("StockCode","nunique"),
    first_purchase=("InvoiceDate","min"),
    last_purchase=("InvoiceDate","max")
).reset_index()


In [45]:
threshold = customer["total_revenue"].quantile(0.9)

In [46]:
customer["high_value"] = (customer["total_revenue"] >= threshold).astype(int)

In [47]:
customer["high_value"].value_counts()

high_value
0    3933
1     438
Name: count, dtype: int64

### 4.2 — Measure the imbalance

- What is the class distribution (high-value vs. regular)?
- If a model always predicted "regular," what would its accuracy be?
- Why is accuracy a poor metric here?



In [48]:
customer["high_value"].value_counts()


high_value
0    3933
1     438
Name: count, dtype: int64

In [51]:
customer["high_value"].value_counts(normalize=True)

high_value
0    0.899794
1    0.100206
Name: proportion, dtype: float64

In [54]:
baseline_accuracy = (customer["high_value"] == 0).mean()
print(round(baseline_accuracy,2))

0.9


The class distribution shows that high-value customers represent about 10% of the dataset, while the remaining 90% are regular customers. This indicates a clear class imbalance.

If a model always predicted the majority class (regular), it would achieve an accuracy of about 90%. However, this model would fail to identify any high-value customers.

Therefore, accuracy is a poor metric in this case. Metrics such as precision, recall, and F1-score are more appropriate for evaluating performance on the minority class.

### 4.3 — Apply resampling

Split the customer-level dataset into train and test sets (80/20). Then apply two resampling techniques to the **training set only**:

1. **Random oversampling** of the minority class
2. **Random undersampling** of the majority class

For each:
- Print the class distribution before and after resampling
- Train a simple model (e.g., `LogisticRegression` or `DecisionTreeClassifier`) on both the original and resampled training sets
- Evaluate on the **original (not resampled) test set** using precision, recall, and F1 for the high-value class


In [61]:
from sklearn.model_selection import train_test_split

X = customer.drop(columns=[
    "CustomerID",
    "high_value",
    "first_purchase",
    "last_purchase"
])

y = customer["high_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [62]:
y_train.value_counts()

high_value
0    3162
1     334
Name: count, dtype: int64

In [63]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

print("Before:")
print(y_train.value_counts())

print("After oversampling:")
print(y_train_over.value_counts())

Before:
high_value
0    3162
1     334
Name: count, dtype: int64
After oversampling:
high_value
0    3162
1    3162
Name: count, dtype: int64


In [64]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("After undersampling:")
print(y_train_under.value_counts())

After undersampling:
high_value
0    334
1    334
Name: count, dtype: int64


In [65]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [66]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       771
           1       1.00      1.00      1.00       104

    accuracy                           1.00       875
   macro avg       1.00      1.00      1.00       875
weighted avg       1.00      1.00      1.00       875



## Task 5: Data Leakage — Introduce, Detect, and Fix

### 5.1 — Intentionally introduce temporal leakage

The dataset spans December 2010 through December 2011. Suppose you want to predict whether a customer will make a purchase in December 2011 based on their behavior in earlier months.

First, do it **wrong**: randomly split all customer features (computed from the full date range) into train and test sets. Train a model and record its performance.




In [76]:
X = customer.drop(columns=["CustomerID","first_purchase","last_purchase"])
y = (customer["last_purchase"].dt.month == 12).astype(int)


In [77]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [78]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [79]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.98      0.92       724
           1       0.74      0.23      0.35       151

    accuracy                           0.85       875
   macro avg       0.80      0.60      0.63       875
weighted avg       0.84      0.85      0.82       875



### 5.2 — Detect the leakage

- Check whether train and test sets contain features computed from overlapping time periods.
- Look for suspiciously high model performance (a sign of leakage).
- Compute feature-target correlations and identify any that seem too good to be true.


In [80]:
print(df_["InvoiceDate"].min())
print(df_["InvoiceDate"].max())

2010-12-01 08:26:00
2011-12-09 12:50:00


In [81]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.98      0.92       724
           1       0.74      0.23      0.35       151

    accuracy                           0.85       875
   macro avg       0.80      0.60      0.63       875
weighted avg       0.84      0.85      0.82       875



In [82]:
corr = customer.drop(columns=["CustomerID"]).corr()
print(corr["last_purchase"].sort_values(ascending=False))

last_purchase     1.000000
num_products      0.302857
first_purchase    0.271438
num_orders        0.259782
high_value        0.230333
total_revenue     0.132148
Name: last_purchase, dtype: float64


### 5.3 — Fix with a correct temporal split

Now do it **right**:
- Use data from December 2010 through September 2011 to compute customer features (the "observation window").
- Use data from October 2011 through December 2011 to create the target variable: did the customer make at least one purchase in this "prediction window"?
- Train the same model and compare performance.

```python
# Temporal split
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[df["InvoiceDate"] <= observation_end]
df_pred = df[df["InvoiceDate"] >= prediction_start]
```


In [85]:
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df_[df_["InvoiceDate"] <= observation_end].copy()
df_pred = df_[df_["InvoiceDate"] >= prediction_start].copy()

In [86]:
df_obs["Revenue"] = df_obs["Quantity"] * df_obs["UnitPrice"]

customer_features = df_obs.groupby("CustomerID").agg(
    total_revenue=("Revenue","sum"),
    num_orders=("InvoiceNo","nunique"),
    num_products=("StockCode","nunique")
).reset_index()

In [87]:
target = df_pred.groupby("CustomerID").size().reset_index(name="purchases")

target["dec_purchase"] = (target["purchases"] > 0).astype(int)

target = target[["CustomerID","dec_purchase"]]

In [88]:
data = customer_features.merge(target, on="CustomerID", how="left")

data["dec_purchase"] = data["dec_purchase"].fillna(0)

In [89]:
X = data.drop(columns=["CustomerID","dec_purchase"])
y = data["dec_purchase"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [90]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.63      0.77      0.69       350
         1.0       0.74      0.58      0.65       380

    accuracy                           0.67       730
   macro avg       0.68      0.68      0.67       730
weighted avg       0.69      0.67      0.67       730



To avoid temporal leakage, customer features were computed using data from December 2010 to September 2011 (observation window). The target variable was created using purchases from October 2011 to December 2011 (prediction window). The model was then trained and evaluated using this temporally correct setup.